# 02 - ARMA-GARCH(1,1) Fitting & Rolling Window Innovations

**Project:** ARMA-GARCH Beta Risk Model  
**Author:** Amos Anderson  
**Purpose:** Fit ARMA(1,1)-GARCH(1,1) to log returns for each asset,
diagnose the fit, and extract standardised innovations via a 250-period
rolling estimation window across 1,000 assessment periods.

The innovations output from this notebook are the direct input to
`03_nig_fitting.ipynb` for heavy-tail distribution fitting.

In [1]:
# Imports

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "iframe"

import warnings
warnings.filterwarnings("ignore")

from src.data_utils import load_processed, save_processed
print("Imports OK")

Imports OK


In [2]:
# Load returns

returns = load_processed("returns.parquet")
TICKERS = list(returns.columns)
print(f"Loaded: {returns.shape[0]} periods × {len(TICKERS)} assets")
print(f"Date range: {returns.index[0].date()} → {returns.index[-1].date()}")
print(f"Tickers: {TICKERS}")

Loaded from C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\returns.parquet  shape=(5047, 5)
Loaded: 5047 periods × 5 assets
Date range: 2005-01-04 → 2025-12-30
Tickers: ['AAPL', 'EURUSD=X', 'GLD', 'TIP', '^GSPC']


----

## Rolling window design

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| Estimation window | 250 periods | ~1 trading year; project's standard; enough to identify GARCH parameters robustly |
| Assessment window | 1,000 periods | ~4 trading years; provides sufficient exceedances for binomial testing |
| Total periods used | 1,250 | Well within our 5,047 available |
| Step size | 1 day | True out-of-sample: each prediction uses only past data |


The rolling window slides forward one day at a time. At each step:
> 1. Fit ARMA(1,1)-GARCH(1,1) on the 250-period estimation window
> 2. Extract the one-step-ahead conditional mean $\mu_{t+1}$ and volatility $\sigma_{t+1}$ forecasts
> 3. Fit NIG to the 250 in-sample standardised innovations $\nu_t = \epsilon_t / \sigma_t$, where $\epsilon_t = r_t - (c + \phi r_{t-1} + \theta \epsilon_{t-1})$ is the ARMA residual
> 4. Compute VaR and CVaR via the affine transform: $R_{t+1} \sim \text{NIG}(\hat\alpha, \hat\beta, \mu_{t+1} + \hat\mu_{\text{NIG}}, \sigma_{t+1} \cdot \hat\delta_{\text{NIG}})$
> 5. Compute the PIT value $u_t = F_{\text{NIG}_t}\!\bigl((r_{t+1} - \mu_{t+1})/\sigma_{t+1}\bigr)$ for model evaluation
> 6. Store all outputs and slide forward one day

In [3]:
# Implement src/arma_garch.py

import importlib
import src.arma_garch as _ag
importlib.reload(_ag)
from src.arma_garch import fit_arma_garch, rolling_window_innovations, ljung_box_test
print("src.arma_garch imported OK")

src.arma_garch imported OK


In [4]:
# Single-window diagnostic (S&P 500)
## Before running the full loop, we verify the fit on one window manually.

ticker = "^GSPC"
window_data = returns[ticker].iloc[:250].values

result = fit_arma_garch(window_data)

print(f"Converged: {result.converged}")
print(f"\nFitted parameters:")
for k, v in result.params.items():
    print(f"  {k:20s}: {v:.6f}")

# One-step-ahead forecasts (Kaufman slide 33)
print(f"\nOne-step-ahead forecasts:")
print(f"  μ_forecast (conditional mean):  {result.mu_forecast:.6f}")
print(f"  σ_forecast (conditional vol):   {result.sigma_forecast:.6f}")
print(f"  σ_forecast annualised:          {result.sigma_forecast * np.sqrt(252):.4f}")

print(f"\nIn-sample innovations — mean: {result.innovations.mean():.4f}  "
      f"std: {result.innovations.std():.4f}")
print("Expected: mean ≈ 0, std ≈ 1")

Converged: True

Fitted parameters:
  Const               : 0.035383
  y[1]                : -0.063190
  omega               : 0.037274
  alpha[1]            : 0.105693
  beta[1]             : 0.797522

One-step-ahead forecasts:
  μ_forecast (conditional mean):  0.000601
  σ_forecast (conditional vol):   0.005232
  σ_forecast annualised:          0.0831

In-sample innovations — mean: -0.0220  std: 0.9978
Expected: mean ≈ 0, std ≈ 1


In [5]:
# Ljung box test on single window

lb = ljung_box_test(result.innovations)

print("Ljung-Box test on single estimation window (^GSPC, first 250 days)")
print(f"\n  Innovations    - stat: {lb['lb_stat_innov']:.3f}  "
      f"p-value: {lb['lb_pval_innov']:.4f}")
print(f"  Innovations²   - stat: {lb['lb_stat_sq']:.3f}  "
      f"p-value: {lb['lb_pval_sq']:.4f}")
print("\n  p > 0.05 on both implies ARMA-GARCH removed autocorrelation and ARCH effects")
print("  p < 0.05 on innovations² implies residual ARCH effects remain (flag this)")

Ljung-Box test on single estimation window (^GSPC, first 250 days)

  Innovations    - stat: 6.974  p-value: 0.7279
  Innovations²   - stat: 9.321  p-value: 0.5020

  p > 0.05 on both implies ARMA-GARCH removed autocorrelation and ARCH effects
  p < 0.05 on innovations² implies residual ARCH effects remain (flag this)


----

## Full rolling window for all assets

Running 1,000 one-step-ahead predictions per asset.
Each window: fit ARMA(1,1)-GARCH(1,1) on 250 periods -> extract $sigma $ forecast ->
compute innovation for the next day.

**This will take several minutes.** Progress prints every 100 windows.

In [6]:
# Running the rolling window loop

In [7]:
ESTIMATION_WINDOW = 250
ASSESSMENT_WINDOW = 1000

all_results = {}

for ticker in TICKERS:
    print(f"\n{'='*50}")
    print(f"Processing: {ticker}")
    print(f"{'='*50}")

    df = rolling_window_innovations(
        returns[ticker],
        estimation_window=ESTIMATION_WINDOW,
        assessment_window=ASSESSMENT_WINDOW,
    )
    all_results[ticker] = df

print("\nAll assets complete.")


Processing: AAPL
  100/1000 windows complete (GARCH fails: 4, NIG fails: 4)
  200/1000 windows complete (GARCH fails: 52, NIG fails: 52)
  300/1000 windows complete (GARCH fails: 52, NIG fails: 52)
  400/1000 windows complete (GARCH fails: 52, NIG fails: 52)
  500/1000 windows complete (GARCH fails: 52, NIG fails: 52)
  600/1000 windows complete (GARCH fails: 52, NIG fails: 52)
  700/1000 windows complete (GARCH fails: 52, NIG fails: 52)
  800/1000 windows complete (GARCH fails: 52, NIG fails: 52)
  900/1000 windows complete (GARCH fails: 52, NIG fails: 52)
  1000/1000 windows complete (GARCH fails: 52, NIG fails: 52)

Done. 1000 predictions.
  GARCH convergence failures: 52 (5.2%)
  NIG convergence failures:   52 (5.2%)

Processing: EURUSD=X
  100/1000 windows complete (GARCH fails: 0, NIG fails: 0)
  200/1000 windows complete (GARCH fails: 45, NIG fails: 45)
  300/1000 windows complete (GARCH fails: 78, NIG fails: 78)
  400/1000 windows complete (GARCH fails: 78, NIG fails: 78)
  50

In [8]:
# Save rolling results

# Save each asset's FULL rolling results (17 columns including NIG, T, VaR, PIT)
for ticker, df in all_results.items():
    safe_name = ticker.replace("^", "").replace("=", "_")
    save_processed(df, f"rolling_{safe_name}.parquet")

# Combined DataFrames for easy access in downstream notebooks
innovations_combined = pd.DataFrame({
    ticker: all_results[ticker]["innovation"]
    for ticker in TICKERS
})
save_processed(innovations_combined, "innovations.parquet")

sigma_combined = pd.DataFrame({
    ticker: all_results[ticker]["sigma_forecast"]
    for ticker in TICKERS
})
save_processed(sigma_combined, "sigma_forecasts.parquet")

mu_combined = pd.DataFrame({
    ticker: all_results[ticker]["mu_forecast"]
    for ticker in TICKERS
})
save_processed(mu_combined, "mu_forecasts.parquet")

# PIT values — all three distributions
for pit_name in ["pit_nig", "pit_t", "pit_gauss"]:
    pit_df = pd.DataFrame({
        ticker: all_results[ticker][pit_name]
        for ticker in TICKERS
    })
    save_processed(pit_df, f"{pit_name}.parquet")

# VaR and CVaR
for measure in ["var_95", "var_99", "cvar_95", "cvar_99"]:
    m_df = pd.DataFrame({
        ticker: all_results[ticker][measure]
        for ticker in TICKERS
    })
    save_processed(m_df, f"{measure}.parquet")

print("\nSaved: rolling_*.parquet + combined innovations, sigma, mu, PIT (×3), VaR, CVaR")

Saved to C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\rolling_AAPL.parquet
Saved to C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\rolling_EURUSD_X.parquet
Saved to C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\rolling_GLD.parquet
Saved to C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\rolling_TIP.parquet
Saved to C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\rolling_GSPC.parquet
Saved to C:\Users\

In [9]:
innovations_combined

,AAPL,EURUSD=X,GLD,TIP,^GSPC
date,,,,,
2021-11-05,0.216550,-1.594178,1.550436,0.867525,0.411807
2021-11-08,-0.577225,0.315862,0.289927,0.822961,-0.060956
2021-11-09,0.116255,0.703441,0.579374,1.561575,-0.834987
2021-11-10,-1.798301,0.197009,1.312101,-0.834428,-1.574243
2021-11-11,-0.192627,-3.076475,0.553430,-0.158420,-0.126708
...,...,...,...,...,...
2025-12-23,0.458300,1.951162,1.028871,0.111471,0.488820
2025-12-24,0.323205,0.833000,-0.544114,0.931723,0.325271
2025-12-26,-0.237607,-0.442135,0.915884,-0.126734,-0.130114


In [10]:
# Plot: Conditional volatility

fig = make_subplots(
    rows=len(TICKERS), cols=1,
    shared_xaxes=True,
    subplot_titles=TICKERS,
    vertical_spacing=0.06,
)

colors = px.colors.qualitative.Plotly

for i, ticker in enumerate(TICKERS):
    sigma = all_results[ticker]["sigma_forecast"] * np.sqrt(252)  # annualised

    fig.add_trace(
        go.Scatter(
            x=sigma.index,
            y=sigma.values,
            mode="lines",
            name=ticker,
            line=dict(color=colors[i], width=0.8),
            showlegend=True,
        ),
        row=i+1, col=1,
    )
    fig.update_yaxes(title_text="Ann. vol", row=i+1, col=1)

fig.update_layout(
    title="GARCH(1,1) Conditional Volatility - All Assets (Annualised)",
    height=900,
    template="plotly_white",
    hovermode="x unified",
)
fig.write_html("../outputs/figures/02_conditional_volatility.html")
fig.show()
print("Figure saved to outputs/figures/02_conditional_volatility.html")

Figure saved to outputs/figures/02_conditional_volatility.html


In [11]:
# Standardised Innovations

fig = make_subplots(
    rows=len(TICKERS), cols=1,
    shared_xaxes=True,
    subplot_titles=TICKERS,
    vertical_spacing=0.06,
)

for i, ticker in enumerate(TICKERS):
    innov = all_results[ticker]["innovation"]

    fig.add_trace(
        go.Scatter(
            x=innov.index,
            y=innov.values,
            mode="lines",
            name=ticker,
            line=dict(color=colors[i], width=0.6),
            showlegend=True,
        ),
        row=i+1, col=1,
    )
    fig.add_hline(y=0, line_dash="dash", line_color="black",
                  line_width=0.8, row=i+1, col=1)
    fig.update_yaxes(title_text="Innovation", row=i+1, col=1)

fig.update_layout(
    title="Standardised Innovations After ARMA-GARCH Filtering",
    height=900,
    template="plotly_white",
    hovermode="x unified",
)
fig.write_html("../outputs/figures/02_innovations.html")
fig.show()
print("Figure saved to outputs/figures/02_innovations.html")

Figure saved to outputs/figures/02_innovations.html


In [12]:
# Ljung-box across all assets

lb_rows = []

for ticker in TICKERS:
    innov = all_results[ticker]["innovation"].values
    lb    = ljung_box_test(innov)
    lb_rows.append({
        "Ticker":               ticker,
        "LB stat (innovations)":   round(lb["lb_stat_innov"], 3),
        "p-value (innovations)":   round(lb["lb_pval_innov"], 4),
        "LB stat (innov²)":        round(lb["lb_stat_sq"], 3),
        "p-value (innov²)":        round(lb["lb_pval_sq"], 4),
        "Autocorrelation removed": "✓" if lb["lb_pval_innov"] > 0.05 else "✗",
        "ARCH effects removed":    "✓" if lb["lb_pval_sq"]    > 0.05 else "✗",
    })

lb_df = pd.DataFrame(lb_rows).set_index("Ticker")
print("Ljung-Box Diagnostic Results (lag=10)\n")
display(lb_df)
#print(lb_df.to_string())

Ljung-Box Diagnostic Results (lag=10)



,LB stat (innovations),p-value (innovations),LB stat (innov²),p-value (innov²),Autocorrelation removed,ARCH effects removed
Ticker,,,,,,
AAPL,9.519,0.4837,30.592,0.0007,✓,✗
EURUSD=X,7.492,0.6783,6.911,0.7338,✓,✓
GLD,7.685,0.6596,23.638,0.0086,✓,✗
TIP,22.712,0.0119,13.179,0.2138,✗,✓
^GSPC,9.431,0.4917,20.452,0.0253,✓,✗


In [13]:
# Plot: Innovation distributions vs Gaussian

fig = make_subplots(rows=1, cols=len(TICKERS),
                    subplot_titles=TICKERS,
                    horizontal_spacing=0.05)

x_range = np.linspace(-6, 6, 400)
gaussian = (1 / np.sqrt(2 * np.pi)) * np.exp(-0.5 * x_range**2)

for i, ticker in enumerate(TICKERS):
    innov = all_results[ticker]["innovation"].dropna().values

    fig.add_trace(
        go.Histogram(
            x=innov,
            nbinsx=80,
            histnorm="probability density",
            name=ticker,
            marker_color=colors[i],
            opacity=0.65,
            showlegend=True,
        ),
        row=1, col=i+1,
    )
    fig.add_trace(
        go.Scatter(
            x=x_range,
            y=gaussian,
            mode="lines",
            name="Gaussian" if i == 0 else None,
            showlegend=(i == 0),
            line=dict(color="black", width=1.5),
        ),
        row=1, col=i+1,
    )
    fig.update_xaxes(title_text="Innovation", range=[-6, 6], row=1, col=i+1)

fig.update_layout(
    title="Innovation Distributions vs Standard Gaussian",
    height=420,
    template="plotly_white",
)
fig.write_html("../outputs/figures/02_innovation_distributions.html")
fig.show()
print("Figure saved to outputs/figures/02_innovation_distributions.html")

Figure saved to outputs/figures/02_innovation_distributions.html


In [14]:
# Per-window NIG parameter evolution
# Each window has its own NIG(α, β, μ, δ) fitted to 250 in-sample innovations.

fig = make_subplots(
    rows=4, cols=1,
    shared_xaxes=True,
    subplot_titles=["α (tail heaviness)", "β (asymmetry)", "μ (location)", "δ (scale)"],
    vertical_spacing=0.08,
)

for i, ticker in enumerate(TICKERS):
    df = all_results[ticker]
    for j, param in enumerate(["nig_alpha", "nig_beta", "nig_mu", "nig_delta"]):
        fig.add_trace(
            go.Scatter(
                x=df.index, y=df[param],
                mode="lines", name=ticker if j == 0 else None,
                showlegend=(j == 0),
                line=dict(color=colors[i], width=0.8),
            ),
            row=j+1, col=1,
        )

fig.update_layout(
    title="Per-Window NIG Parameters — Rolling Estimation",
    height=800,
    template="plotly_white",
    hovermode="x unified",
)
fig.write_html("../outputs/figures/02_nig_params_evolution.html")
fig.show()
print("Figure saved to outputs/figures/02_nig_params_evolution.html")

Figure saved to outputs/figures/02_nig_params_evolution.html


The NIG parameters vary across windows — each prediction date has its own $\text{NIG}(\alpha_t, \beta_t, \mu_t, \delta_t)$. This is the **corrected** methodology: NIG is fitted per-window inside the rolling loop, not once globally on pooled innovations.

- **$\alpha$** controls tail heaviness (smaller = heavier tails)
- **$\beta$** controls asymmetry (negative = left skew, typical for equities)
- **$\mu$** is the NIG location parameter
- **$\delta$** is the NIG scale parameter

In [15]:
# Summary statistics of Innovations

from src.data_utils import summary_statistics

innov_df = pd.DataFrame({
    ticker: all_results[ticker]["innovation"]
    for ticker in TICKERS
})

stats = summary_statistics(innov_df)
print("Innovation Summary Statistics\n")
print(stats.round(4).to_string())
print("\nNote: excess kurtosis >> 0 confirms heavy tails remain after GARCH filtering.")
print("This justifies NIG over Gaussian for the innovation distribution in notebook 03.")

Innovation Summary Statistics

                      AAPL   EURUSD=X        GLD        TIP      ^GSPC
mean (daily)       -0.0232     0.0061     0.0359    -0.0167    -0.0440
std (daily)         1.0914     1.0548     1.0620     0.9984     1.0488
mean (annual)      -5.8401     1.5250     9.0404    -4.2129   -11.0841
vol (annual)       17.3262    16.7446    16.8593    15.8493    16.6499
skewness           -0.2415     0.0718    -0.0262    -0.0685    -0.5016
excess kurtosis     4.5009     2.9526     1.4017     1.7735     1.8483
min                -7.6045    -5.6223    -4.4285    -4.4282    -5.4275
max                 4.8326     6.6073     5.2512     5.0495     3.7316
obs              1000.0000  1000.0000  1000.0000  1000.0000  1000.0000

Note: excess kurtosis >> 0 confirms heavy tails remain after GARCH filtering.
This justifies NIG over Gaussian for the innovation distribution in notebook 03.


In [16]:
# Verification

innov_check = load_processed("innovations.parquet")
sigma_check = load_processed("sigma_forecasts.parquet")
pit_check   = load_processed("pit_values.parquet")
var99_check = load_processed("var99.parquet")

n_predictions = innov_check.shape[0]
n_expected    = ASSESSMENT_WINDOW

# Allow up to 5% GARCH failures (windows where first window fails get skipped)
assert n_predictions >= n_expected * 0.95, \
    f"Too few predictions: {n_predictions} vs expected {n_expected}"
assert innov_check.isna().sum().sum() == 0, \
    "NaNs found in innovations"

print(f"Verification PASSED")
print(f"  Predictions:      {n_predictions} / {n_expected} expected")
print(f"  Innovations:      {innov_check.shape}")
print(f"  Sigma forecasts:  {sigma_check.shape}")
print(f"  PIT values:       {pit_check.shape}")
print(f"  VaR 99%:          {var99_check.shape}")
print(f"  Date range:       {innov_check.index[0].date()} to {innov_check.index[-1].date()}")

# Show per-window NIG params for first asset
first_ticker = TICKERS[0]
safe = first_ticker.replace("^", "").replace("=", "_")
rolling_check = load_processed(f"rolling_{safe}.parquet")
print(f"\nRolling results for {first_ticker}: {rolling_check.shape[1]} columns")
print(f"Columns: {list(rolling_check.columns)}")
nig_conv_rate = rolling_check["nig_converged"].mean() * 100
print(f"NIG convergence rate: {nig_conv_rate:.1f}%")

Loaded from C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\innovations.parquet  shape=(1000, 5)
Loaded from C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\sigma_forecasts.parquet  shape=(1000, 5)
Loaded from C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\pit_values.parquet  shape=(1000, 5)
Loaded from C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\var99.parquet  shape=(1000, 5)
Verification PASSED
  Predictions:      1000 / 1000 expected
  Innovations:      (1000, 5)
  Sigma forecasts:  (1000, 5)
  PIT values:       (1

----

## Handoff to notebook 03

Saved to `data/processed/`:

- `innovations.parquet` -- 1,000 $\times $ 5 standardised innovations (ε_t = r_t / σ_t)
- `sigma_forecasts.parquet` -- 1,000 $\times $ 5 one-step-ahead conditional volatilities
- `rolling_{ticker}.parquet` -- full per-asset rolling results including raw returns

**Notebook 03** fits Normal Inverse Gaussian (NIG) and Student T distributions
to these innovations, comparing both against a Gaussian baseline.
The NIG parameters + $\sigma $ forecasts feed directly into VaR/CVaR computation
in notebook 04.